# Introduction to NLP Fundamentals in TensorFlow

NLP has the goal of deriving information out of natural language (could be sequences of text or speech).

Another common term for NLP problems is sequence to sequence problems (seq2seq).

In [1]:
# ============================================================================
# ⚡ GEEKOM A9 MAX - Setup Rápido para Estudos
# Cole no início de qualquer notebook e execute primeiro
# ============================================================================

import os, warnings
os.environ.update({
    'TF_CPP_MIN_LOG_LEVEL': '2',           # Menos logs
    'TF_ENABLE_ONEDNN_OPTS': '1',          # oneDNN (2-3x mais rápido)
    'OMP_NUM_THREADS': '32',               # 32 threads
    'MKL_NUM_THREADS': '32',
    'TF_NUM_INTEROP_THREADS': '4',
    'TF_NUM_INTRAOP_THREADS': '32',
    'TF_XLA_FLAGS': '--tf_xla_auto_jit=2', # XLA JIT
    'CUDA_VISIBLE_DEVICES': '-1',          # CPU only
})
warnings.filterwarnings('ignore')

import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Config TensorFlow
try: 
    tf.config.optimizer.set_jit(True)
    tf.config.threading.set_inter_op_parallelism_threads(4)
    tf.config.threading.set_intra_op_parallelism_threads(32)
except: pass
tf.config.set_visible_devices([], 'GPU')

# Gráficos
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = [14, 6]

print(f"✅ TensorFlow {tf.__version__} | Ryzen 9 (32t) | 96GB RAM | oneDNN ativo")
print("="*80)


✅ TensorFlow 2.15.0 | Ryzen 9 (32t) | 96GB RAM | oneDNN ativo


In [2]:
# Import series of helper functions for the notebook
from helper_functions import unzip_data, create_tensorboard_callback, plot_loss_curves, compare_historys

## Get a text dataset

The dataset we're going to be using is Kaggle's introduction to NLP dataset (text samples of Tweets labelled as disaster or not disaster).

In [3]:
# import zipfile
# import urllib.request

# # Baixar o arquivo ZIP
# url = "https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip"
# zip_path = "nlp_getting_started.zip"
# urllib.request.urlretrieve(url, zip_path)

# # # Descompactar o arquivo ZIP
# with zipfile.ZipFile(zip_path, "r") as zip_ref:
#     zip_ref.extractall()

## Visualization of the data (text)

Once you've acquired a new dataset to work with, what should you do first?

Explore it? Inspect it? Verify it? Become one with it?

All correct.

Remember the motto: visualize, visualize, visualize.

Right now, our text data samples are in the form of '.csv' files. For an easy way to make them visual, let's turn them into pandas DataFrame's.

📖 Reading: You might come across text datasets in many different formats. Aside from CSV files (what we're working with), you'll probably encounter '.txt' files and '.json' files too. In this section, we'll learn how to read these types of files as well. For working with these type of files, I'd recommend reading the two following articles by RealPython:

* [How to Read and Write Files in Python](https://realpython.com/read-write-files-python/)
* [How to Read and Write JSON Files in Python](https://realpython.com/python-json/)

In [4]:
import pandas as pd
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
print(train_df.head())

   id keyword location                                               text  \
0   1     NaN      NaN  Our Deeds are the Reason of this #earthquake M...   
1   4     NaN      NaN             Forest fire near La Ronge Sask. Canada   
2   5     NaN      NaN  All residents asked to 'shelter in place' are ...   
3   6     NaN      NaN  13,000 people receive #wildfires evacuation or...   
4   7     NaN      NaN  Just got sent this photo from Ruby #Alaska as ...   

   target  
0       1  
1       1  
2       1  
3       1  
4       1  


In [5]:
# Shuffle training dataframe
train_df_shuffled = train_df.sample(frac=1, random_state=42) # random_state is set to 42 for reproducibility
train_df_shuffled.head()

,id,keyword,location,text,target
2644,3796,destruction,NaN,So you have a new weapon that can cause un-ima...,1
2227,3185,deluge,NaN,The f$&amp;@ing things I do for #GISHWHES Just...,0
5448,7769,police,UK,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe...,1
132,191,aftershock,NaN,Aftershock back to school kick off was great. ...,0
6845,9810,trauma,"Montgomery County, MD",in response to trauma Children of Addicts deve...,0


In [6]:
# What does the test dataframe look like?
test_df.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


In [7]:
# How many examples of each class are in the training set?
train_df.target.value_counts()

target
0    4342
1    3271
Name: count, dtype: int64

In [8]:
# How many total samples?
len(train_df), len(test_df)

(7613, 3263)

In [9]:
# Let's visualize some random training examples
import random
random_index = random.randint(0, len(train_df)-5)
for row in train_df_shuffled[["text", "target"]][random_index:random_index+5].itertuples():
    _, text, target = row
    print(f"Target: {target}", "(real threat)" if target > 0 else "(not a threat)")
    print(f"Text:\n{text}\n")
    print("---\n")

Target: 1 (real threat)
Text:
Body shops inundated with cars dented by hail... Good news insurance pays... Bad news :  you are stuck with deductible !
#wcvb

---

Target: 0 (not a threat)
Text:
Err:509

---

Target: 0 (not a threat)
Text:
How do people bake without turning their kitchen into a war zone of eggs and flour

---

Target: 1 (real threat)
Text:
@KlaraJoelsson Well I have seen it now! That's a bummer. We've had this heat wave tho... 43'c!! I'd prefer the rain... :P

---

Target: 0 (not a threat)
Text:
[Latest Post] Bayelsa poll: Tension in Bayelsa as Patience Jonathan plans to hijack APC PDP http://t.co/B2yvLMPepR

---



### Split data into training and valiation sets

In [10]:
# split the data into training and testing sets
from sklearn.model_selection import train_test_split

In [11]:
# Use train_test_split to split the data into training and testing sets.
train_sentences, val_sentences, train_labels, val_labels = train_test_split(train_df_shuffled["text"].to_numpy(),
                                                                            train_df_shuffled["target"].to_numpy(),
                                                                            test_size=0.1,  # 10% of the data
                                                                            random_state=42) # random seed for reproducibility

In [12]:
# Check the length of the text.
len(train_sentences), len(train_labels), len(val_sentences), len(val_labels)

(6851, 6851, 762, 762)

In [13]:
# Check  the first 10 samples
train_sentences[:10], train_labels[:10]

(array(['@mogacola @zamtriossu i screamed after hitting tweet',
        'Imagine getting flattened by Kurt Zouma',
        '@Gurmeetramrahim #MSGDoing111WelfareWorks Green S welfare force ke appx 65000 members har time disaster victim ki help ke liye tyar hai....',
        "@shakjn @C7 @Magnums im shaking in fear he's gonna hack the planet",
        'Somehow find you and I collide http://t.co/Ee8RpOahPk',
        '@EvaHanderek @MarleyKnysh great times until the bus driver held us hostage in the mall parking lot lmfao',
        'destroy the free fandom honestly',
        'Weapons stolen from National Guard Armory in New Albany still missing #Gunsense http://t.co/lKNU8902JE',
        '@wfaaweather Pete when will the heat wave pass? Is it really going to be mid month? Frisco Boy Scouts have a canoe trip in Okla.',
        'Patient-reported outcomes in long-term survivors of metastatic colorectal cancer - British Journal of Surgery http://t.co/5Yl4DC1Tqt'],
       dtype=object),
 array([0,

## Converting text into numbers

When dealing with a text problem, one of the first things you'll have to do before you can build a model is to convert your text to numbers.
There are a few ways to do this, namely:
- Tokenization – direct mapping of a token (a token could be a word or a character) to a number
- Embedding – create a matrix of feature vectors for each token (the size of the feature vector can be defined and this embedding can be learned)

In [14]:
# convert text to numbers.
from tensorflow.keras.layers.experimental.preprocessing import TextVectorization

text_vectorizer = TextVectorization(max_tokens=10000, # how many words in the vocabulary (automatically add <OOV>)
                                    standardize="lower_and_strip_punctuation", # convert text to lowercase and remove punctuation
                                    split="whitespace", # split text into words or characters
                                    ngrams=None, # create groups of words or characters
                                    output_mode="int", # how to convert tokens to numbers
                                    output_sequence_length=None, # how long is the output sequence
                                    pad_to_max_tokens=True) # pad the sequence to the max length 

In [15]:
len(train_sentences[0].split()) # how many words in the first tweet

7

In [16]:
# Finde the average numbers of tokens (words) in the training tweets

round(sum([len(tweet.split()) for tweet in train_sentences])/len(train_sentences))

15

In [17]:
# Setup text Vectorization variables
max_vocab_length = 10000 # how many unique words in the vocabulary
max_length = 15 # max length of a text to consider

text_vectorizer = TextVectorization(max_tokens=max_vocab_length,
                                    output_mode="int",
                                    output_sequence_length=max_length)

In [18]:
# Fit the text vectorizer on the training tweets
text_vectorizer.adapt(train_sentences)


In [19]:
# Create a sample sentence and tokenize it
sample_sentence = " There's a flood in my street!"
text_vectorizer([sample_sentence])

<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[264,   3, 232,   4,  13, 698,   0,   0,   0,   0,   0,   0,   0,
          0,   0]], dtype=int64)>

In [20]:
# Choose a random sentence from the training dataset and tokenize it
random_sentence = random.choice(train_sentences)   
print(f"Original tweet: \n {random_sentence}\
      \n\nVectorized Version:")
text_vectorizer([random_sentence])

Original tweet: 
 @DannyRaynard not bad personally I'd get rid of either hazard or aguero for a better striker than berahino      

Vectorized Version:


<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[   1,   34,  281, 4931,  508,   52, 3464,    6, 1416,  423,   53,
           1,   10,    3,  441]], dtype=int64)>

In [21]:
# get the unique words in the vocabulary
words_in_vocab = text_vectorizer.get_vocabulary() # get all the unique words in the vocabulary
top_5_words = words_in_vocab[:5] # get the top 5 words in the vocabulary
bottom_5_words = words_in_vocab[-5:] # get the bottom 5 words in the vocabulary
print(f"Numbers of words in vocab: {len(words_in_vocab)}")
print(f"5 most commom words: {top_5_words}")
print(f"5 least commom words: {bottom_5_words}")

Numbers of words in vocab: 10000
5 most commom words: ['', '[UNK]', 'the', 'a', 'in']
5 least commom words: ['pages', 'paeds', 'pads', 'padres', 'paddytomlinson1']


### Creating and Embedding using Embedding Layer

Creating an Embedding using an Embedding Layer
We've got a way to map our text to numbers. How about we go a step further and turn those numbers into an embedding?

The powerful thing about an embedding is it can be learned during training. This means rather than just being static (e.g. 1 = I, 2 = love, 3 = TensorFlow), a word's numeric representation can be improved as a model goes through data samples.

We can see what an embedding of a word looks like by using the [tf.keras.layers.Embedding laye](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding)r.

The main parameters we're concerned about here are:

* `input_dim` - The size of the vocabulary (e.g. len(text_vectorizer.get_vocabulary()).
* `output_dim` - The size of the output embedding vector, for example, a value of 100 outputs a feature vector of size 100 for each word.
* `embeddings_initializer` - How to initialize the embeddings matrix, default is "uniform" which randomly initalizes embedding matrix with uniform distribution. This can be changed for using pre-learned embeddings.
* `input_length` - Length of sequences being passed to embedding layer.
  
Knowing these, let's make an embedding layer.

In [22]:
from tensorflow.keras import layers

embedding = layers.Embedding(input_dim=max_vocab_length,
                            output_dim=128,
                            input_length=max_length,
                            )

In [23]:
# get a random sentence from the training set
random_sentence = random.choice(train_sentences)
print(f"Original tweet: \n {random_sentence}\
            \n\nEmbedded Version:")

# Embed the radon sentence (turn it into a dense vector of fixed sizes)
sample_embed = embedding(text_vectorizer([random_sentence]))
sample_embed 

Original tweet: 
 Please allow me to reiterate it's not the weapon it's the mindset of the individual! #professional #help! -LEGION! https://t.co/2lGTZkwMqW            

Embedded Version:


<tf.Tensor: shape=(1, 15, 128), dtype=float32, numpy=
array([[[-0.02021451, -0.01477443,  0.04738188, ..., -0.04949293,
         -0.02101883, -0.01105066],
        [ 0.01083039,  0.01141823,  0.04515871, ..., -0.04938312,
          0.01116204, -0.00446742],
        [-0.00906331, -0.03194581,  0.0135137 , ...,  0.01326141,
         -0.03172716,  0.0283596 ],
        ...,
        [-0.0280061 , -0.04868773,  0.02204109, ..., -0.01830186,
          0.01281675, -0.00689526],
        [-0.02931329,  0.01856119, -0.04212996, ...,  0.00858068,
          0.0047853 , -0.00401722],
        [-0.01160326, -0.03917596,  0.03444156, ..., -0.0450161 ,
         -0.01103449, -0.00243767]]], dtype=float32)>

In [24]:
# Check out a single token's embedding
sample_embed[0][0], sample_embed[0][0].shape, random_sentence

(<tf.Tensor: shape=(128,), dtype=float32, numpy=
 array([-0.02021451, -0.01477443,  0.04738188, -0.00790798,  0.04588199,
        -0.01910108,  0.02939036,  0.02485437,  0.00203665,  0.00356247,
        -0.02742314,  0.01515788, -0.02618461, -0.01985877, -0.01013218,
         0.01971331, -0.02058423,  0.04359284, -0.01201241,  0.0110658 ,
        -0.03076079,  0.01276932,  0.01174201, -0.01000658, -0.04664109,
        -0.04179945, -0.04604738, -0.02972095, -0.00470396,  0.04776371,
         0.03427965,  0.00580604, -0.03816988, -0.01115052,  0.01101384,
        -0.03653625, -0.01496691,  0.04579647,  0.00434196, -0.0299746 ,
         0.03007052,  0.00413195,  0.01891668,  0.0360013 ,  0.03805201,
         0.0087567 ,  0.02292484,  0.00253033, -0.01872001,  0.02738884,
        -0.00425913, -0.01552166,  0.0242558 ,  0.0474186 ,  0.04383456,
         0.00829562,  0.02150813, -0.04045679,  0.00042671,  0.00435583,
        -0.00052582,  0.00504807,  0.04706272,  0.01940688, -0.01555425,
  

### Modelling a text dataset (running a series of experiments)
Now we've got a way to turn our text sequences into numbers, it's time to start building a series of modelling experiments.
We'll start with a baseline and move on from there.
- Model 0: Naive Bayes (baseline), this is from Sklearn ML map: https://scikit-learn.org/stable/tutorial/machine_learning_map/index.html (scikit-learn.org in Bing)
- Model 1: Feed-forward neural network (dense model)
- Model 2: LSTM model (RNN)
- Model 3: GRU model (RNN)
- Model 4: Bidirectional-LSTM model (RNN)
- Model 5: 1D Convolutional Neural Network (CNN)
- Model 6: TensorFlow Hub Pretrained Feature Extractor (using transfer learning for NLP)
- Model 7: Same as model 6 with 10% of training data


How are we going to approach all of these?
Use the standard steps in modelling with TensorFlow:
- Create a model
- Build a model
- Fit a model
- Evaluate our model


### Model 0: Getting a baseline

As with all machine learning modelling experiments, it's important to create a baseline model so you've got a benchmark for future experiments to build upon.
To create our baseline, we'll use Sklearn's Multinomial Naive Bayes using the TF‑IDF formula to convert our words to numbers.
> 🔑 Note: It's common practice to use non‑DL algorithms as a baseline because of their speed and then later use DL to see if you can improve upon them.



In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
# Create the pipeline

model_0 = Pipeline([
    ('tfidf', TfidfVectorizer()), # convert text to numbers
    ('clf', MultinomialNB()) # model the text
])

# Fit the pipeline to the training data
model_0.fit(train_sentences, train_labels)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())])

In [26]:
# Evaluate our baseline model
baseline_score = model_0.score(train_sentences, train_labels)
print(f"Our baseline model achieves an accuracy of: {baseline_score:.3f}%")

Our baseline model achieves an accuracy of: 0.887%


In [27]:
# Make predictions
baseline_preds = model_0.predict(val_sentences)
baseline_preds[:20]

array([1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1],
      dtype=int64)

### Creating an evaluation function for our model experiments
We could evaluate these as they are but since we're going to be evaluating several models in the same way going forward, let's create a helper function which takes an array of predictions and ground truth labels and computes the following:

* Accuracy
* Precision
* Recall
* F1-score
> 🔑 Note: Since we're dealing with a classification problem, the above metrics are the most appropriate. If we were working with a regression problem, other metrics such as MAE (mean absolute error) would be a better choice. We can find more at: https://scikit-learn.org/0.16/modules/model_evaluation.html

In [28]:
# Function to evaluate: accuracy, precision, recall, f1-score
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def calculate_results(y_true, y_pred):
  """
  Calculates model accuracy, precision, recall and f1 score of a binary classification model.

  Args:
  -----
  y_true = true labels in the form of a 1D array
  y_pred = predicted labels in the form of a 1D array

  Returns a dictionary of accuracy, precision, recall, f1-score.
  """
  # Calculate model accuracy
  model_accuracy = accuracy_score(y_true, y_pred) * 100
  # Calculate model precision, recall and f1 score using "weighted" average
  model_precision, model_recall, model_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted")
  model_results = {"accuracy": model_accuracy,
                  "precision": model_precision,
                  "recall": model_recall,
                  "f1": model_f1}
  return model_results

In [29]:
# get baseline results
baseline_results = calculate_results(y_true=val_labels, y_pred=baseline_preds)
baseline_results

{'accuracy': 79.26509186351706,
 'precision': 0.8111390004213173,
 'recall': 0.7926509186351706,
 'f1': 0.7862189758049549}

### Model 1: Simple dense model

In [30]:
# Create a tensorboard callback (need to create a new one for each model)
from helper_functions import create_tensorboard_callback

# create a directory to save TensorBoard logs
SAVE_DIR = "model_logs"

In [31]:
from tensorflow.keras import layers

inputs = layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)       # (None, 1) → (None, sequence_length)
x = embedding(x)                  # (None, sequence_length) → (None, sequence_length, embed_dim)
x = layers.GlobalAveragePooling1D()(x)  # (None, sequence_length, embed_dim) → (None, embed_dim)
outputs = layers.Dense(1, activation="sigmoid")(x)  # (None, embed_dim) → (None, 1)

model_1 = tf.keras.Model(inputs, outputs, name="model_1_dense")

In [32]:
model_1.summary()

Model: "model_1_dense"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_1 (Text  (None, 15)                0         
 Vectorization)                                                  
                                                                 
 embedding (Embedding)       (None, 15, 128)           1280000   
                                                                 
 global_average_pooling1d (  (None, 128)               0         
 GlobalAveragePooling1D)                                         
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 1280129 (4.88 MB)
Trainable params: 128

In [33]:
# Compile the model
model_1.compile(
    loss="binary_crossentropy",  # NOT sparse_categorical or from_logits=True
    optimizer="adam",
    metrics=["accuracy"]
)

In [34]:
model_1.summary()

Model: "model_1_dense"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_1 (Text  (None, 15)                0         
 Vectorization)                                                  
                                                                 
 embedding (Embedding)       (None, 15, 128)           1280000   
                                                                 
 global_average_pooling1d (  (None, 128)               0         
 GlobalAveragePooling1D)                                         
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 1280129 (4.88 MB)
Trainable params: 128

In [35]:
# Check intermediate shapes
print("After vectorizer:", text_vectorizer(tf.constant(["test"])).shape)
print("Model summary:")
model_1.summary()

After vectorizer: (1, 15)
Model summary:
Model: "model_1_dense"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_1 (Text  (None, 15)                0         
 Vectorization)                                                  
                                                                 
 embedding (Embedding)       (None, 15, 128)           1280000   
                                                                 
 global_average_pooling1d (  (None, 128)               0         
 GlobalAveragePooling1D)                                         
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params

In [36]:
# Fit the model
model_1_history = model_1.fit(x=train_sentences, # input sentences can be a list of strings due to text preprocessing layer built-in model
                              y=train_labels,
                              epochs=5,
                              validation_data=(val_sentences, val_labels),
                              callbacks=[create_tensorboard_callback(dir_name=SAVE_DIR, 
                                                                     experiment_name="simple_dense_model")])

Saving TensorBoard log files to: model_logs/simple_dense_model/20260407-180739
Epoch 1/5

215/215 [==============================] - 2s 5ms/step - loss: 0.6123 - accuracy: 0.6981 - val_loss: 0.5378 - val_accuracy: 0.7480
Epoch 2/5
215/215 [==============================] - 1s 4ms/step - loss: 0.4421 - accuracy: 0.8178 - val_loss: 0.4762 - val_accuracy: 0.7795
Epoch 3/5
215/215 [==============================] - 1s 4ms/step - loss: 0.3478 - accuracy: 0.8578 - val_loss: 0.4597 - val_accuracy: 0.7861
Epoch 4/5
215/215 [==============================] - 1s 4ms/step - loss: 0.2846 - accuracy: 0.8917 - val_loss: 0.4663 - val_accuracy: 0.7874
Epoch 5/5
215/215 [==============================] - 1s 4ms/step - loss: 0.2377 - accuracy: 0.9108 - val_loss: 0.4830 - val_accuracy: 0.7874


In [37]:
# Check the results
model_1.evaluate(val_sentences, val_labels)

24/24 [==============================] - 0s 1ms/step - loss: 0.4830 - accuracy: 0.7874


[0.482974112033844, 0.787401556968689]

In [38]:
embedding.weights

[<tf.Variable 'embedding/embeddings:0' shape=(10000, 128) dtype=float32, numpy=
 array([[ 0.01849023,  0.05306024,  0.01627181, ...,  0.00099015,
          0.02382394,  0.01911984],
        [ 0.03669391, -0.01869809, -0.01394112, ..., -0.00364794,
          0.03148883, -0.0195371 ],
        [-0.01329188,  0.03516423, -0.05746272, ..., -0.01016137,
          0.02059248,  0.01180529],
        ...,
        [ 0.04080344, -0.01318628,  0.00275508, ..., -0.01101116,
         -0.02496024,  0.03565336],
        [ 0.06095303,  0.04929765,  0.01380067, ..., -0.01745885,
          0.01302747,  0.04047225],
        [ 0.08212652,  0.02018759, -0.05228037, ..., -0.02825366,
          0.06224365,  0.08185074]], dtype=float32)>]

In [39]:
# Make some predictions with our new model and evaluate those predictions
model_1_pred_probs = model_1.predict(val_sentences)
model_1_pred_probs.shape

24/24 [==============================] - 0s 840us/step


(762, 1)

In [40]:
model_1_pred_probs[0]

array([0.302523], dtype=float32)

In [41]:
model_1_pred_probs[:10]

array([[0.302523  ],
       [0.70439583],
       [0.99766916],
       [0.1221485 ],
       [0.10755331],
       [0.936721  ],
       [0.9077632 ],
       [0.99296844],
       [0.95935655],
       [0.28557393]], dtype=float32)

In [42]:
# Convert prediction probabilities to labels
model_1_preds = tf.round(model_1_pred_probs)  # rounds 0.5+ to 1, below 0.5 to 0
model_1_preds.shape  # (762, 1) — still fine

# Flatten if needed for evaluation
model_1_preds = tf.squeeze(model_1_preds)  # (762,) — matches val_labels shape
model_1_preds.shape  # (762,)

TensorShape([762])

In [43]:
from sklearn.metrics import classification_report

print(classification_report(val_labels, model_1_preds))

              precision    recall  f1-score   support

           0       0.76      0.89      0.82       414
           1       0.83      0.67      0.74       348

    accuracy                           0.79       762
   macro avg       0.80      0.78      0.78       762
weighted avg       0.79      0.79      0.78       762



In [44]:
# Convert model prediction probabilities to label format
model_1_preds = tf.squeeze(tf.round(model_1_pred_probs))
model_1_preds[:20]

<tf.Tensor: shape=(20,), dtype=float32, numpy=
array([0., 1., 1., 0., 0., 1., 1., 1., 1., 0., 0., 1., 0., 0., 0., 0., 0.,
       0., 0., 0.], dtype=float32)>

In [45]:
# Calculate our model_1 results
from pyexpat import model


model_1_results = calculate_results(y_true=val_labels, y_pred=model_1_preds)
print(model_1_results)

{'accuracy': 78.74015748031496, 'precision': 0.7942180127180873, 'recall': 0.7874015748031497, 'f1': 0.7838012115396069}


In [46]:
baseline_results

{'accuracy': 79.26509186351706,
 'precision': 0.8111390004213173,
 'recall': 0.7926509186351706,
 'f1': 0.7862189758049549}

In [47]:
import numpy as np
np.array(list(model_1_results.values()) > np.array(list(baseline_results.values())))


array([False, False, False, False])

## Visualize learned embeddings

In [48]:
# get the vocabulary from the text vectorization
words_in_vocab = text_vectorizer.get_vocabulary()
len(words_in_vocab), words_in_vocab[:20]

(10000,
 ['',
  '[UNK]',
  'the',
  'a',
  'in',
  'to',
  'of',
  'and',
  'i',
  'is',
  'for',
  'on',
  'you',
  'my',
  'with',
  'it',
  'that',
  'at',
  'by',
  'this'])

In [49]:
# Model1 summary
model_1.summary()

Model: "model_1_dense"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_1 (Text  (None, 15)                0         
 Vectorization)                                                  
                                                                 
 embedding (Embedding)       (None, 15, 128)           1280000   
                                                                 
 global_average_pooling1d (  (None, 128)               0         
 GlobalAveragePooling1D)                                         
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 1280129 (4.88 MB)
Trainable params: 128

In [50]:
# Get the weight matrix of embedding layer
# (these are the numerical representations of each token in our training data, wich have been learned for 5 epochs)
embed_weights = model_1.get_layer('embedding').get_weights()[0]
print(embed_weights.shape) # same size as vocab size and embedding dim (output dim of our embedding layer)

(10000, 128)


Now we've got the embedding matrix our model has learned to represent our tokens, let's see how we can visualize it.

To do so, TensorFlow has a handy tool called projector: http://projector.tensorflow.org/

And TensorFlow also has an incredible guide on word embeddings themselves: https://www.tensorflow.org/text/tutorials/word_embeddings

In [51]:
# Create embedding files (we got this from TensorFlo's word embedding documentation )

import io
out_v = io.open('vectors.tsv', 'w', encoding='utf-8') # write vectors.tsv
out_m = io.open('metadata.tsv', 'w', encoding='utf-8') # write metadata.tsv

for index, word in enumerate(words_in_vocab):
  if index == 0:
    continue  # skip 0, it's padding.
  vec = embed_weights[index]
  out_v.write('\t'.join([str(x) for x in vec]) + "\n")
  out_m.write(word + "\n")
out_v.close()
out_m.close()


## Recurrent Neural Networks (RNN's)

Recurrent Neural Networks (RNN's)

RNN's are useful for sequence data.

The premise of a recurrent neural network is to use the representation of a previous input to aid the representation of a later input.

If you want an overview of the internals of a recurrent neural network, see the following:
- MIT's sequence modelling lecture https://youtu.be/qjrad0V0uJE
- Chris Olah's intro to LSTMs: https://colah.github.io/posts/2015-08-Understanding-LSTMs/
- Andrej Karpathy's the unreasonable effectiveness of recurrent neural networks: http://karpathy.github.io/2015/05/21/rnn-effectiveness/

### Model 2: LSTM

LSTM = long short term memory (one of the most popular LSTM cells)

Our structure of an RNN typically looks like this:

```
Input (text) -> Tokenize -> Embedding -> Layers (RNNs/dense) -> Output (label probability)
```

In [52]:
# Create an LSTM model
from tensorflow.keras import layers
inputs = layers.Input(shape=(1,), dtype='string')
x = text_vectorizer(inputs)
x = embedding(x)
# print(x.shape)
# x = layers.LSTM(64, return_sequences=True)(x) # when you're stacking RNN cells together, you will need to return_sequences=True 
# print(x.shape)
x = layers.LSTM(64)(x)
# print(x.shape)
# x = layers.Dense(64, activation='relu')(x)
# print(x.shape)
outputs = layers.Dense(1, activation='sigmoid')(x)
model_2 = tf.keras.Model(inputs, outputs, name='model_2_LSTM')

In [53]:
# get a summary of the model
model_2.summary()

Model: "model_2_LSTM"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_1 (Text  (None, 15)                0         
 Vectorization)                                                  
                                                                 
 embedding (Embedding)       (None, 15, 128)           1280000   
                                                                 
 lstm (LSTM)                 (None, 64)                49408     
                                                                 
 dense_1 (Dense)             (None, 1)                 65        
                                                                 
Total params: 1329473 (5.07 MB)
Trainable params: 1329473 (5.07 MB)
Non-trainable params: 0 (0.00 Byte)
________________

In [54]:
# Compile the model
model_2.compile(loss='binary_crossentropy', 
                optimizer='adam', 
                metrics=['accuracy'])

In [55]:
# Fit the model
model_2_history = model_2.fit(train_sentences,
                                train_labels, 
                                epochs=5, 
                                validation_data=(val_sentences, val_labels),
                                callbacks=[create_tensorboard_callback(SAVE_DIR,
                                                                    'model_2_LSTM')])

Saving TensorBoard log files to: model_logs/model_2_LSTM/20260407-180746
Epoch 1/5
215/215 [==============================] - 3s 9ms/step - loss: 0.2220 - accuracy: 0.9232 - val_loss: 0.5166 - val_accuracy: 0.7861
Epoch 2/5
215/215 [==============================] - 2s 8ms/step - loss: 0.1563 - accuracy: 0.9415 - val_loss: 0.6083 - val_accuracy: 0.7861
Epoch 3/5
215/215 [==============================] - 2s 8ms/step - loss: 0.1285 - accuracy: 0.9512 - val_loss: 0.6585 - val_accuracy: 0.7835
Epoch 4/5
215/215 [==============================] - 2s 8ms/step - loss: 0.1055 - accuracy: 0.9597 - val_loss: 0.7450 - val_accuracy: 0.7874
Epoch 5/5
215/215 [==============================] - 2s 8ms/step - loss: 0.0849 - accuracy: 0.9663 - val_loss: 0.9465 - val_accuracy: 0.7756


In [56]:
# make predictions with LSTM model
model_2_pred_probs = model_2.predict(val_sentences)
model_2_pred_probs[:10]

24/24 [==============================] - 0s 2ms/step


array([[2.9485044e-03],
       [8.0955344e-01],
       [9.9956518e-01],
       [4.4732034e-02],
       [3.2875221e-04],
       [9.9651283e-01],
       [9.7026372e-01],
       [9.9970120e-01],
       [9.9947882e-01],
       [5.4748851e-01]], dtype=float32)

In [57]:
# Convert model 2 pred probs to labels
model_2_preds = tf.squeeze(tf.round(model_2_pred_probs))
model_2_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 1.], dtype=float32)>

In [58]:
# Calculate model 2 results
model_2_results = calculate_results(y_true=val_labels, 
                                    y_pred=model_2_preds)
model_2_results

{'accuracy': 77.55905511811024,
 'precision': 0.7780461459912817,
 'recall': 0.7755905511811023,
 'f1': 0.7732287214395843}

### Model 3: GRU
Another popular and effective RNN component is the GRU, or gated recurrent unit. The GRU cell has similar features to an LSTM cell but has fewer parameters.


In [59]:
# Build an RNN using the GRU cell
from tensorflow.keras import layers
inputs = layers.Input(shape=(1, ), dtype=tf.string)
x = text_vectorizer(inputs)
x = embedding(x)
# x = layers.GRU(64, return_sequences=True)(x) # if you want to stack recurrent layers on top of each other, you need to set return_sequences=True
# x = layers.LSTM(64, return_sequences=True)(x)
x = layers.GRU(64)(x)
# x = layers.Dense(64, activation="relu")(x)
# x = layers.GlobalAveragePooling1D()(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model_3 = tf.keras.Model(inputs, outputs, name="model_3_GRU")

In [60]:
model_3.summary()

Model: "model_3_GRU"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_1 (Text  (None, 15)                0         
 Vectorization)                                                  
                                                                 
 embedding (Embedding)       (None, 15, 128)           1280000   
                                                                 
 gru (GRU)                   (None, 64)                37248     
                                                                 
 dense_2 (Dense)             (None, 1)                 65        
                                                                 
Total params: 1317313 (5.03 MB)
Trainable params: 1317313 (5.03 MB)
Non-trainable params: 0 (0.00 Byte)
_________________

In [61]:
# Compile the model
model_3.compile(loss="binary_crossentropy", 
                optimizer="adam", 
                metrics=["accuracy"])

In [62]:
# Fit the model
model_3_history = model_3.fit(train_sentences, 
                              train_labels, 
                              epochs=5, 
                              validation_data=(val_sentences, val_labels), 
                              callbacks=[create_tensorboard_callback(SAVE_DIR, 
                                                                     "model_3_GRU")])


Saving TensorBoard log files to: model_logs/model_3_GRU/20260407-180756
Epoch 1/5
215/215 [==============================] - 3s 9ms/step - loss: 0.1546 - accuracy: 0.9397 - val_loss: 0.6738 - val_accuracy: 0.7743
Epoch 2/5
215/215 [==============================] - 2s 7ms/step - loss: 0.0842 - accuracy: 0.9670 - val_loss: 0.7756 - val_accuracy: 0.7782
Epoch 3/5
215/215 [==============================] - 2s 7ms/step - loss: 0.0743 - accuracy: 0.9720 - val_loss: 0.8461 - val_accuracy: 0.7848
Epoch 4/5
215/215 [==============================] - 2s 8ms/step - loss: 0.0633 - accuracy: 0.9749 - val_loss: 1.0899 - val_accuracy: 0.7690
Epoch 5/5
215/215 [==============================] - 2s 8ms/step - loss: 0.0571 - accuracy: 0.9764 - val_loss: 1.3807 - val_accuracy: 0.7703


In [63]:
# make some predicitions with our GRU model
model_3_pred_probs = model_3.predict(val_sentences)
model_1_preds[:10]

24/24 [==============================] - 0s 2ms/step


<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 0.], dtype=float32)>

In [64]:
# Convert model 3 pred probs to labels
model_3_preds = tf.squeeze(tf.round(model_3_pred_probs))
model_3_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 1.], dtype=float32)>

In [65]:
# Calculate model 3 results
model_3_results = calculate_results(y_true=val_labels, y_pred=model_3_preds)
model_3_results

{'accuracy': 77.03412073490814,
 'precision': 0.7743290047665735,
 'recall': 0.7703412073490814,
 'f1': 0.7671895343790688}

### Model 4: Bidirectional RNN
Normal RNNs go from left to right (just like you'd read an English sentence). However, a bidirectional RNN goes from right to left as well as left to right.


In [66]:
# Build a bidirectional RNN in TensorFlow
from tensorflow.keras import layers
inputs = layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)
x = embedding(x)
#x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
x = layers.Bidirectional(layers.LSTM(64))(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model_4 = tf.keras.Model(inputs, outputs, name="model_4_bidirectional")

In [67]:
model_4.summary()

Model: "model_4_bidirectional"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_4 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_1 (Text  (None, 15)                0         
 Vectorization)                                                  
                                                                 
 embedding (Embedding)       (None, 15, 128)           1280000   
                                                                 
 bidirectional (Bidirection  (None, 128)               98816     
 al)                                                             
                                                                 
 dense_3 (Dense)             (None, 1)                 129       
                                                                 
Total params: 1378945 (5.26 MB)
Trainable par

In [68]:
# Compile model
model_4.compile(loss="binary_crossentropy", 
                optimizer=tf.keras.optimizers.Adam(), 
                metrics=["accuracy"])

In [69]:
# Fit the model
model_4_history = model_4.fit(train_sentences,
                              train_labels,
                              validation_data=(val_sentences, val_labels),
                              epochs=5,
                              callbacks=[create_tensorboard_callback("SAVE_DIR", "model_4_bidirectional")])

Saving TensorBoard log files to: SAVE_DIR/model_4_bidirectional/20260407-180806
Epoch 1/5
215/215 [==============================] - 4s 11ms/step - loss: 0.1039 - accuracy: 0.9707 - val_loss: 0.9981 - val_accuracy: 0.7717
Epoch 2/5
215/215 [==============================] - 2s 9ms/step - loss: 0.0491 - accuracy: 0.9778 - val_loss: 1.2834 - val_accuracy: 0.7769
Epoch 3/5
215/215 [==============================] - 2s 9ms/step - loss: 0.0451 - accuracy: 0.9801 - val_loss: 1.2769 - val_accuracy: 0.7612
Epoch 4/5
215/215 [==============================] - 2s 9ms/step - loss: 0.0442 - accuracy: 0.9787 - val_loss: 1.4135 - val_accuracy: 0.7585
Epoch 5/5
215/215 [==============================] - 2s 9ms/step - loss: 0.0398 - accuracy: 0.9819 - val_loss: 1.4088 - val_accuracy: 0.7690


In [70]:
# make predictions with our bidirectional model
model_4_pred_probs = model_4.predict(val_sentences)
model_4_pred_probs[:10]

24/24 [==============================] - 0s 2ms/step


array([[1.8767867e-03],
       [7.4227130e-01],
       [9.9998325e-01],
       [1.1886483e-01],
       [3.6556961e-05],
       [9.9913871e-01],
       [2.5736153e-01],
       [9.9998778e-01],
       [9.9998057e-01],
       [9.8493624e-01]], dtype=float32)

In [71]:
# Convert pred probs to pred labels
model_4_preds = tf.squeeze(tf.round(model_4_pred_probs))
model_4_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 0., 1., 1., 1.], dtype=float32)>

In [72]:
# Calculate the results of our bidirectional model
model_4_results = calculate_results(y_true=val_labels, y_pred=model_4_preds)
model_4_results

{'accuracy': 76.9028871391076,
 'precision': 0.772410916855732,
 'recall': 0.7690288713910761,
 'f1': 0.7660901868079131}

### Convolution Neural Networks for Text (and other types of sequences)
We've used CNNs for images but images are typically 2D (height × width)… however, our text data is 1D.
Previously we've used Conv2D for our image data but now we're going to use Conv1D.
The typical structure of a Conv1D model for sequences (in our case, text):

```
Inputs (text) → Tokenization → Embedding → Layer(s) (typically Conv1D + pooling) → Outputs (class probabilities)
```

### Model 5: Conv1D

- For different exmplanations of parameters see:
https://poloclub.github.io/cnn-explainer/ (for 2D but can relate to 1D)
- Difference between "same" and "valid" padding:
https://stackoverflow.com/questions/37674306/what-is-the-difference-between-same-and-valid-padding-in-tf-nn-max-pool-of-t 

In [73]:
# test out our embedding layer, Conv1D layer and max_pooling layer
embedding_test = embedding(text_vectorizer(["Hello world"])) # turn target sequence into embedding
conv_1D = layers.Conv1D(filters=64, # 
                        kernel_size=5, # kernel_size is the size of the convolutional window (5 words at a time)
                        strides=1,
                        activation="relu",
                        padding="same") # padding="valid" means that the output will be smaller than the input, "same" is the same of the input between layers
conv_1D_output = conv_1D(embedding_test) # apply the convolutional layer to the embedding layer output
max_pool = layers.GlobalMaxPool1D() # max_pooling layer
max_pool_output = max_pool(conv_1D_output) # apply the max_pooling layer to the convolutional layer output the most important features from the sequence.


embedding_test.shape, conv_1D_output.shape, max_pool_output.shape

(TensorShape([1, 15, 128]), TensorShape([1, 15, 64]), TensorShape([1, 64]))

In [ ]:
# embedding_test



In [75]:
# conv_1D_output

In [ ]:
# max_pool_output

In [79]:
# Create 1-dimensional convolutional layer to model sequences
from tensorflow.keras import layers
inputs = layers.Input(shape=(1,), dtype= tf.string)
x = text_vectorizer(inputs)
x = embedding(x)
x = layers.Conv1D(filters=64, kernel_size=5, strides=1, activation='relu', padding="valid")(x)
x = layers.GlobalMaxPool1D()(x)
# x = layers.Dense(64, activation='relu')
outputs = layers.Dense(1, activation='sigmoid')(x)
model_5 = tf.keras.Model(inputs, outputs, name="model_5_Conv1D")

# Compile Conv1D
model_5.compile(loss='binary_crossentropy', 
                optimizer='adam', 
                metrics=['accuracy'])

#Get a summary of our model
model_5.summary()


Model: "model_5_Conv1D"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_8 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_1 (Text  (None, 15)                0         
 Vectorization)                                                  
                                                                 
 embedding (Embedding)       (None, 15, 128)           1280000   
                                                                 
 conv1d_3 (Conv1D)           (None, 11, 64)            41024     
                                                                 
 global_max_pooling1d_3 (Gl  (None, 64)                0         
 obalMaxPooling1D)                                               
                                                                 
 dense_6 (Dense)             (None, 1)              

In [80]:
# Fit the model
from gc import callbacks


model_5_history = model_5.fit(train_sentences, 
                              train_labels,
                              epochs=5,
                              validation_data=(val_sentences, val_labels),
                              callbacks=[create_tensorboard_callback(SAVE_DIR, "Conv1D")])

Saving TensorBoard log files to: model_logs/Conv1D/20260407-182703
Epoch 1/5
215/215 [==============================] - 2s 6ms/step - loss: 0.1227 - accuracy: 0.9604 - val_loss: 0.9028 - val_accuracy: 0.7638
Epoch 2/5
215/215 [==============================] - 1s 5ms/step - loss: 0.0747 - accuracy: 0.9714 - val_loss: 1.0212 - val_accuracy: 0.7743
Epoch 3/5
215/215 [==============================] - 1s 5ms/step - loss: 0.0624 - accuracy: 0.9759 - val_loss: 1.1315 - val_accuracy: 0.7585
Epoch 4/5
215/215 [==============================] - 1s 5ms/step - loss: 0.0543 - accuracy: 0.9780 - val_loss: 1.1790 - val_accuracy: 0.7625
Epoch 5/5
215/215 [==============================] - 1s 5ms/step - loss: 0.0505 - accuracy: 0.9785 - val_loss: 1.2746 - val_accuracy: 0.7598


In [81]:
# make some predictions with our Conv1D model
model_5_pred_probs = model_5.predict(val_sentences)
model_5_pred_probs[:10]

24/24 [==============================] - 0s 1ms/step


array([[5.7868417e-02],
       [9.0535837e-01],
       [9.9983501e-01],
       [3.4718107e-02],
       [6.9328522e-08],
       [9.7557956e-01],
       [8.8527799e-01],
       [9.9996018e-01],
       [9.9999881e-01],
       [8.7326431e-01]], dtype=float32)

In [83]:
# Convert model 5 pred probs to labels
model_5_preds = tf.squeeze(tf.round(model_5_pred_probs))
model_5_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 1.], dtype=float32)>

In [86]:
# Evaluate model 5 predictions
model_5_results = calculate_results(y_true=val_labels, 
                                    y_pred=model_5_preds)
model_5_results

{'accuracy': 75.98425196850394,
 'precision': 0.7612947135580711,
 'recall': 0.7598425196850394,
 'f1': 0.7575964683921443}

In [87]:
baseline_results

{'accuracy': 79.26509186351706,
 'precision': 0.8111390004213173,
 'recall': 0.7926509186351706,
 'f1': 0.7862189758049549}

## Model 6: tensorflow Hub pretrained Sentence Encoder

Now we've built a few of our own models, let's try and use transfer learning for NLP, specifically using TensorFlow Hub's Universal Sentence Encoder: https://tfhub.dev/google/universal-sentence-encoder/4
See how the USE was created here: https://arxiv.org/abs/1803.11175

In [90]:
sample_sentence

" There's a flood in my street!"

In [88]:
import tensorflow_hub as hub
embed = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")
embed_samples = embed([sample_sentence, "When you can the universal sentence encoder on a sentence, it turns it into numbers."])
print(embed_samples[0][:50])

tf.Tensor(
[-0.01157022  0.02485909  0.02878049 -0.01271502  0.03971541  0.0882776
  0.02680985  0.05589838 -0.01068732 -0.00597293  0.00639324 -0.01819521
  0.00030815  0.0910589   0.05874645 -0.03180628  0.01512474 -0.05162928
  0.00991369 -0.06865346 -0.04209307  0.0267898   0.0301101   0.00321071
 -0.00337968 -0.04787361  0.02266718 -0.00985926 -0.04063615 -0.01292094
 -0.04666382  0.05630298 -0.03949255  0.00517684  0.02495828 -0.0701444
  0.02871509  0.04947679 -0.00633974 -0.08960192  0.02807122 -0.00808364
 -0.01360597  0.05998648 -0.10361788 -0.05195372  0.00232956 -0.02332528
 -0.03758107  0.03327729], shape=(50,), dtype=float32)


In [92]:
embed_samples[0].shape

TensorShape([512])

In [93]:
# Create a keras layer using the USE (Universal Sentence Encoder) pretrained layer from tensorflow hub
sentence_encoder_layer = hub.KerasLayer("https://tfhub.dev/google/universal-sentence-encoder/4", 
                                        input_shape=[], 
                                        dtype=tf.string, 
                                        trainable=False,
                                        name="USE")

In [95]:
# Create model using the Sequential API
from os import name


model_6 = tf.keras.Sequential([
    sentence_encoder_layer,
    # layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid'),
],  name="model_6_USE")

# Compile model
model_6.compile(loss='binary_crossentropy',
                optimizer='adam',
                metrics=['accuracy'])

model_6.summary()

Model: "model_6_USE"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 USE (KerasLayer)            (None, 512)               256797824 
                                                                 
 dense_7 (Dense)             (None, 1)                 513       
                                                                 
Total params: 256798337 (979.61 MB)
Trainable params: 513 (2.00 KB)
Non-trainable params: 256797824 (979.61 MB)
_________________________________________________________________


In [96]:
# Train a classifier on top of USE pretrained embeddings
model_6_history = model_6.fit(train_sentences,
                              train_labels,
                              epochs=5,
                              validation_data=(val_sentences, val_labels),
                              callbacks=[create_tensorboard_callback(SAVE_DIR, "tf_hub_sentence_encoder")])

Saving TensorBoard log files to: model_logs/tf_hub_sentence_encoder/20260407-193853
Epoch 1/5
215/215 [==============================] - 3s 5ms/step - loss: 0.6480 - accuracy: 0.7402 - val_loss: 0.6097 - val_accuracy: 0.7756
Epoch 2/5
215/215 [==============================] - 1s 5ms/step - loss: 0.5806 - accuracy: 0.7905 - val_loss: 0.5604 - val_accuracy: 0.7808
Epoch 3/5
215/215 [==============================] - 1s 5ms/step - loss: 0.5375 - accuracy: 0.7940 - val_loss: 0.5288 - val_accuracy: 0.7835
Epoch 4/5
215/215 [==============================] - 1s 4ms/step - loss: 0.5091 - accuracy: 0.7978 - val_loss: 0.5083 - val_accuracy: 0.7861
Epoch 5/5
215/215 [==============================] - 1s 4ms/step - loss: 0.4891 - accuracy: 0.8005 - val_loss: 0.4935 - val_accuracy: 0.7874


In [97]:
# Make predictions with USE TF Hub model
model_6_pred_probs = model_6.predict(val_sentences)
model_6_pred_probs[:10]

24/24 [==============================] - 0s 5ms/step


array([[0.36762357],
       [0.6807536 ],
       [0.8555087 ],
       [0.32929227],
       [0.65122217],
       [0.7351819 ],
       [0.81645566],
       [0.84376204],
       [0.7540794 ],
       [0.18725577]], dtype=float32)

In [98]:
# Convert prediction probabilities to labels
model_6_preds = tf.squeeze(tf.round(model_6_pred_probs))
model_6_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 1., 1., 1., 1., 1., 0.], dtype=float32)>

In [100]:
# calculate model 6 performance metrics
model_6_results = calculate_results(y_true=val_labels,
                                    y_pred=model_6_preds)
model_6_results

{'accuracy': 78.74015748031496,
 'precision': 0.7877428758427124,
 'recall': 0.7874015748031497,
 'f1': 0.7863300776686603}

In [101]:
baseline_results

{'accuracy': 79.26509186351706,
 'precision': 0.8111390004213173,
 'recall': 0.7926509186351706,
 'f1': 0.7862189758049549}